# MiniLM + FAISS: definitions and cross-reference context

Iteration 17 retains the original fixed-window MiniLM baseline. Indexing resolves unique numbered sections and quoted-term definitions within each contract. Extra context influences ranking; returned evidence remains original source text. This notebook reviews saved train results and makes no external calls.

In [2]:
from pathlib import Path
import json,sys
import pandas as pd
ROOT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p/'src/retrieval.py').exists())
sys.path.insert(0, str(ROOT/'src'))
sys.path.insert(0, str(ROOT/'scripts'))
from project_paths import source_path
rows=[json.loads(line) for line in (ROOT/'experiments/semantic_log.jsonl').read_text(encoding='utf-8').splitlines()]
run=next(row for row in reversed(rows) if row['iteration']==17)
run['config']

{'name': 'dense_minilm_enriched_faiss',
 'retriever_type': 'enriched_dense',
 'base_config': 'semantic_configs/minilm.json',
 'max_context_chars': 900,
 'context_weight': 0.25,
 'enrichment_protocol': 1}

## Indexing design

Each passage keeps its existing embedding. Resolved context is embedded separately and mixed with weight 0.25, then normalized. This avoids replacing passage tokens with context tokens. Missing or ambiguous targets are not guessed; references are not expanded recursively. Both inputs still have the MiniLM 256-wordpiece limit. The parser is heuristic and does not claim complete legal reference resolution.

In [3]:
from reference_enrichment import context_catalog,enrich_chunk
text='Section 1. Definitions.\n"Licensed Content" means the listed films.\nSection 2. Rights.\nLicensed Content may be used subject to Section 1.'
start=text.index('Licensed Content may')
example=dict(text=text[start:],start=start,end=len(text))
enrich_chunk(example,text,context_catalog(text))

{'context': 'Section 1. Definitions.\n"Licensed Content" means the listed films.\n\n"Licensed Content" means the listed films.',
 'sources': [{'start': 0,
   'end': 67,
   'kind': 'section',
   'key': '1',
   'text': 'Section 1. Definitions.\n"Licensed Content" means the listed films.\n'},
  {'start': 24,
   'end': 66,
   'kind': 'definition',
   'key': 'Licensed Content',
   'text': '"Licensed Content" means the listed films.'}],
 'unresolved_sections': []}

In [4]:
table=[]
for name,result in run['metrics'].items():
    m=next(m for m in result['retrieval'] if m['k']==5)
    table.append(dict(version=name,precision=m['precision'],recall=m['recall'],benchmark_precision=m['benchmark_precision'],benchmark_recall=m['benchmark_recall'],hit_rate=m['hit'],correct_abstention=result['correct_abstention'],false_abstention=result['false_abstention']))
pd.DataFrame(table).set_index('version').style.format('{:.2%}')

,precision,recall,benchmark_precision,benchmark_recall,hit_rate,correct_abstention,false_abstention
version,,,,,,,
reference,4.03%,35.75%,4.19%,38.90%,46.43%,0.00%,0.00%
candidate,3.87%,35.70%,4.03%,39.20%,46.43%,0.00%,0.00%


## Abstention thresholds

Report correct and false abstention separately. The prospective limit is 5% false abstention (at most 7/140 train answerable queries). Train-selected thresholds are tuning results, not independent validation. Check precision and recall as well; rejecting everything is not success.

In [5]:
table=[]
for trial in run['threshold_trials']:
    m=next(m for m in trial['metrics']['retrieval'] if m['k']==5)
    table.append(dict(threshold=trial['threshold'],precision=m['precision'],recall=m['recall'],correct_abstention=trial['metrics']['correct_abstention'],false_abstention=trial['metrics']['false_abstention'],accepted=trial['accepted']))
pd.DataFrame(table)

,threshold,precision,recall,correct_abstention,false_abstention,accepted
0,0.40,0.038732,0.357010,0.000000,0.000000,False
1,0.45,0.038732,0.357010,0.000000,0.000000,False
2,0.50,0.038732,0.357010,0.130435,0.000000,True
3,0.55,0.038732,0.357010,0.391304,0.007143,True
4,0.60,0.036351,0.331425,0.652174,0.078571,False
5,0.65,0.025684,0.229076,0.869565,0.371429,False


In [6]:
print('Enrichment accepted:',run['accepted'],run['checks'])
print('Selected cutoff:',run['selected_threshold'])
print('Indexing:',run['embedding_metadata'])
print('Test evaluations:',run['test_evaluation_runs'])
print('Failure categories:',run['diagnosis_after'])

Enrichment accepted: False {'gain': False, 'precision': True, 'recall': True, 'false_abstention': True, 'correct_abstention': True}
Selected cutoff: 0.55
Indexing: {'chunk_count': 1522, 'enriched_chunks': 765, 'context_inputs_over_token_limit': 0, 'context_encode_ms': 24752.463900018483, 'vectors_sha256': '33dcb75464212f2332174d790c8c8b04abe5353aae30d30d2b0d99077137dd2f'}
Test evaluations: 0
Failure categories: {'complete_coverage': 33, 'right_document_wrong_section': 75, 'partial_coverage': 32}


## Reproduce

Run `python manage.py evaluate_enriched` to reproduce the complete train evaluation and cutoff grid using local cached vectors. Edit `semantic_configs/minilm_enriched.json` for a separately logged experiment. `results/iteration_17_enriched/` stores exact context sources, predictions, decisions and every threshold attempt. Hybrid retrieval and reranking have not been added here.